In [2]:
import numpy as np

# --- 1. SAFE USER INPUT SECTION ---
print("🚁 --- Agricultural Drone Path Optimizer Configuration --- 🚁")

try:
    # Input Charging Station
    raw_cs = input("Enter Charging Station coordinates (X Y) [e.g., 0 0]: ").strip().split()
    charging_station = [float(raw_cs[0]), float(raw_cs[1])]

    # Input Farm Regions
    num_regions = int(input("Enter the number of farm regions to visit: "))
    regions_list = [charging_station]

    print(f"Enter the X Y coordinates for each of the {num_regions} regions (one per line):")
    for i in range(num_regions):
        raw_reg = input(f"  Region {i+1} (X Y): ").strip().split()
        regions_list.append([float(raw_reg[0]), float(raw_reg[1])])

    R = np.array(regions_list)

    # Input Drone Specifications
    BATTERY = float(input("Enter Drone Max Battery Capacity (in units of distance/energy): "))

except (IndexError, ValueError) as e:
    print(f"\n❌ Input Error: Please ensure you type two numbers separated by a space (e.g., 10 20).")
    exit()

# --- 2. OPTIMIZATION CORE ---
DIST = np.linalg.norm(R[:, None] - R, axis=-1)  # Precompute distance matrix

def get_cost(vec):
    path = np.argsort(vec) + 1  # Convert continuous values to region indices
    full_path = [0] + list(path) + [0]
    dist = sum(DIST[full_path[i]][full_path[i+1]] for i in range(len(full_path)-1))
    return dist if dist <= BATTERY else 1e6  # High penalty if battery drains out

def gwo(wolves=30, iterations=200):
    # Initialize wolves
    X = np.random.uniform(-10, 10, (wolves, num_regions))

    for t in range(iterations):
        scores = np.array([get_cost(w) for w in X])
        idx = np.argsort(scores)

        # Explicitly extract the 3 distinct best alpha, beta, and delta positions
        a_pos = X[idx[0]].copy()
        b_pos = X[idx[1]].copy()
        d_pos = X[idx[2]].copy()

        a = 2 - t * (2 / iterations)
        for i in range(wolves):
            # Apply GWO position equation element-wise across the dimensions
            A = 2 * a * np.random.rand(3, num_regions) - a
            C = 2 * np.random.rand(3, num_regions)

            X1 = a_pos - A[0] * abs(C[0] * a_pos - X[i])
            X2 = b_pos - A[1] * abs(C[1] * b_pos - X[i])
            X3 = d_pos - A[2] * abs(C[2] * d_pos - X[i])

            X[i] = (X1 + X2 + X3) / 3

    # Final evaluation to find the absolute best wolf
    final_scores = np.array([get_cost(w) for w in X])
    best_idx = np.argmin(final_scores)
    best_alpha_pos = X[best_idx]

    best_path = np.argsort(best_alpha_pos) + 1
    return [0] + list(best_path) + [0], get_cost(best_alpha_pos)

# --- 3. EXECUTION & DISPLAY ---
print("\n🔄 Optimizing path using Grey Wolf Optimization... Please wait...")
route, distance = gwo()

print("\n⚡ --- Optimization Results --- ⚡")
if distance >= 1e6:
    print("❌ Error: No valid path found within the current battery limit! Try increasing battery capacity or reducing region distances.")
else:
    print(f"Optimal Visiting Sequence: {' -> '.join(map(str, route))}")
    print(f"Total Flight Distance:    {distance:.2f} units")
    print(f"Battery Energy Used:      {distance:.2f} / {BATTERY:.2f}")


🚁 --- Agricultural Drone Path Optimizer Configuration --- 🚁
Enter Charging Station coordinates (X Y) [e.g., 0 0]: 10 10
Enter the number of farm regions to visit: 3
Enter the X Y coordinates for each of the 3 regions (one per line):
  Region 1 (X Y): 20 30
  Region 2 (X Y): 50 60
  Region 3 (X Y): 15 80
Enter Drone Max Battery Capacity (in units of distance/energy): 300

🔄 Optimizing path using Grey Wolf Optimization... Please wait...

⚡ --- Optimization Results --- ⚡
Optimal Visiting Sequence: 0 -> 3 -> 2 -> 1 -> 0
Total Flight Distance:    175.28 units
Battery Energy Used:      175.28 / 300.00
